# Notebook 01: Tower Geometry Inference

This notebook infers sector coverage geometry from the antenna catalog.
It produces sector polygons, Voronoi tessellations, and adaptive H3 mappings
for the metropolitan region (R13), along with two paper figures:

- `fig_sectors_r13_overview.pdf` -- sector polygon map
- `fig_resolution_gain_histogram.pdf` -- resolution gain vs. Voronoi baseline

## Setup

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point, Polygon
from shapely.ops import unary_union
import matplotlib.pyplot as plt
import h3
from scipy.spatial import Voronoi
from scipy.spatial.distance import cdist
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

DATA    = Path("../data")
FIGURES = Path("../figures")
FIGURES.mkdir(parents=True, exist_ok=True)
CATALOG = DATA / "tower_catalog.csv"
CENSUS  = DATA / "Cartografia_censo2024_R13/Cartografia_censo2024_R13.gpkg"

R_EARTH_EFF = 8500e3
CRS_GEO     = "EPSG:4674"
CRS_UTM     = "EPSG:32719"

## Load and clean tower catalog

The catalog is expected to have columns: `cell_id`, `bts_id`, `tech`, `admin_code`, `lat`, `lon`, `height`, `azimuth`.

In [ ]:
cat_raw = pd.read_csv(CATALOG)
print(f"Full catalog: {len(cat_raw):,} cells")

# Filter to Region Metropolitana (admin code starting with 13)
cat = cat_raw[cat_raw["admin_code"].astype(str).str.startswith("13")].copy()
print(f"R13 (by code):  {len(cat):,} cells, {cat['bts_id'].nunique():,} BTS sites")

cat = cat.dropna(subset=["lat", "lon"])

# Spatial clip to R13 boundary
comunas = gpd.read_file(CENSUS, layer="Comunal_CPV24")
r13_boundary = comunas.dissolve().geometry.iloc[0]

pts = gpd.GeoSeries(gpd.points_from_xy(cat.lon, cat.lat), index=cat.index, crs=CRS_GEO)
inside = pts.within(r13_boundary)
cat = cat.loc[inside]
print(f"R13 (spatial):  {len(cat):,} cells, {cat['bts_id'].nunique():,} BTS sites")

# Fill missing heights and azimuths
cat["height"] = cat.groupby("bts_id")["height"].transform(
    lambda s: s.fillna(s.median())
).fillna(cat["height"].median())

null_az = cat["azimuth"].isnull()
bts_has_any_az = cat.groupby("bts_id")["azimuth"].transform(lambda s: s.notna().any())
cat.loc[null_az & ~bts_has_any_az, "azimuth"] = 0.0
cat = cat.dropna(subset=["azimuth"])

print(f"After cleaning: {len(cat):,} cells, {cat['bts_id'].nunique():,} BTS sites")

## Infrastructure classification and site merging

In [ ]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import pdist

HEIGHT_THRESHOLD = 10
cat["is_macro"] = cat["height"] >= HEIGHT_THRESHOLD

n_macro = cat["is_macro"].sum()
n_micro = (~cat["is_macro"]).sum()
print(f"Macro cells (h >= {HEIGHT_THRESHOLD}m): {n_macro:,} ({n_macro/len(cat):.0%})")
print(f"Micro cells (h <  {HEIGHT_THRESHOLD}m): {n_micro:,} ({n_micro/len(cat):.0%})")

# Merge co-located BTS (within 20 m)
bts_pos = cat.groupby("bts_id")[["lat", "lon"]].first()
bts_pts = gpd.GeoDataFrame(
    bts_pos, geometry=gpd.points_from_xy(bts_pos.lon, bts_pos.lat), crs=CRS_GEO
).to_crs(CRS_UTM)
coords = np.column_stack([bts_pts.geometry.x, bts_pts.geometry.y])

Z = linkage(pdist(coords), method="complete")
labels = fcluster(Z, t=20, criterion="distance")

bts_pos["_cluster"] = labels
site_rep = bts_pos.groupby("_cluster").apply(lambda g: sorted(g.index)[0])
bts_to_site = bts_pos["_cluster"].map(site_rep)

n_sites = bts_pos["_cluster"].nunique()
print(f"Clustered {len(bts_pos):,} BTS into {n_sites:,} physical sites")

cat["orig_bts_id"] = cat["bts_id"]
cat["bts_id"] = cat["orig_bts_id"].map(bts_to_site)

cell_lookup = cat[["cell_id", "orig_bts_id", "bts_id", "is_macro"]].rename(
    columns={"orig_bts_id": "bts_id_orig", "bts_id": "site_id"}
)

## Sector grouping

In [ ]:
cat_macro = cat[cat["is_macro"]].copy()

cat_macro["azimuth_r"] = (cat_macro["azimuth"] / 5).round() * 5
cat_macro.loc[cat_macro["azimuth_r"] == 360, "azimuth_r"] = 0.0

sectors = (
    cat_macro.groupby(["bts_id", "azimuth_r"])
    .agg(
        lat=("lat", "first"),
        lon=("lon", "first"),
        height=("height", "mean"),
        n_cells=("cell_id", "count"),
        techs=("tech", lambda x: ",".join(sorted(x.unique()))),
        cell_ids=("cell_id", lambda x: list(x)),
        admin_code=("admin_code", "first"),
    )
    .reset_index()
    .rename(columns={"azimuth_r": "azimuth"})
)

n_sectors_per_bts = sectors.groupby("bts_id").size().rename("n_sectors")
sectors = sectors.merge(n_sectors_per_bts, on="bts_id")

print(f"Physical sectors: {len(sectors):,}")
print(f"Sites:            {sectors['bts_id'].nunique():,}")
print(f"\nSectors per site distribution:")
print(n_sectors_per_bts.value_counts().sort_index().to_string())

## Beamwidth and coverage radius

In [ ]:
sectors["beamwidth"] = 360.0 / sectors["n_sectors"]

# BTS-level dataframe
bts = (
    sectors.groupby("bts_id")
    .agg(
        lat=("lat", "first"),
        lon=("lon", "first"),
        n_sectors=("n_sectors", "first"),
        mean_height=("height", "mean"),
        admin_code=("admin_code", "first"),
    )
    .reset_index()
)

bts_gdf = gpd.GeoDataFrame(
    bts, geometry=gpd.points_from_xy(bts.lon, bts.lat), crs=CRS_GEO
).to_crs(CRS_UTM)
bts["x"] = bts_gdf.geometry.x
bts["y"] = bts_gdf.geometry.y

# Nearest-neighbour distance
coords = bts[["x", "y"]].values
dists = cdist(coords, coords)
np.fill_diagonal(dists, np.inf)
bts["d_nn"] = dists.min(axis=1)
bts["d_nn3"] = np.sort(dists, axis=1)[:, :3].mean(axis=1)

# Coverage radius: min of radio horizon and 0.65 * d_nn
bts["r_horizon"] = np.sqrt(2 * bts["mean_height"] * R_EARTH_EFF)
bts["radius"] = np.minimum(bts["r_horizon"], bts["d_nn"] * 0.65)

print("Radius (m):")
print(bts["radius"].describe().to_string())

## Sector polygon construction

In [ ]:
def make_sector_polygon(x, y, azimuth_deg, beamwidth_deg, radius_m, n_points=32):
    if beamwidth_deg >= 360:
        angles = np.linspace(0, 2 * np.pi, n_points + 1)
        ring = [(x + radius_m * np.sin(a), y + radius_m * np.cos(a)) for a in angles]
        return Polygon(ring)

    center_rad = np.deg2rad(90 - azimuth_deg)
    half_bw = np.deg2rad(beamwidth_deg / 2)

    start = center_rad + half_bw
    end   = center_rad - half_bw
    angles = np.linspace(start, end, n_points)
    arc = [(x + radius_m * np.cos(a), y + radius_m * np.sin(a)) for a in angles]

    return Polygon([(x, y)] + arc + [(x, y)])


# Merge radius to sectors and build polygons
sectors = sectors.merge(
    bts[["bts_id", "radius", "d_nn", "d_nn3", "x", "y"]],
    on="bts_id", suffixes=("", "_bts"),
)

polys = []
for _, row in sectors.iterrows():
    poly = make_sector_polygon(
        row["x"], row["y"],
        row["azimuth"], row["beamwidth"], row["radius"],
    )
    polys.append(poly)

sectors_gdf = gpd.GeoDataFrame(sectors, geometry=polys, crs=CRS_UTM)
sectors_gdf["area_km2"] = sectors_gdf.geometry.area / 1e6
sectors_geo = sectors_gdf.to_crs(CRS_GEO)

## Sector centroids

In [ ]:
def sector_centroid(x, y, azimuth_deg, beamwidth_deg, radius_m):
    if beamwidth_deg >= 360:
        return x, y
    alpha = np.deg2rad(beamwidth_deg)
    d_c = (2 * radius_m / 3) * np.sin(alpha / 2) / (alpha / 2)
    angle_rad = np.deg2rad(90 - azimuth_deg)
    return x + d_c * np.cos(angle_rad), y + d_c * np.sin(angle_rad)

centroids = sectors_gdf.apply(
    lambda r: sector_centroid(r["x"], r["y"], r["azimuth"], r["beamwidth"], r["radius"]),
    axis=1, result_type="expand",
)
sectors_gdf["cx"] = centroids[0]
sectors_gdf["cy"] = centroids[1]

centroid_pts = gpd.GeoSeries(
    gpd.points_from_xy(sectors_gdf["cx"], sectors_gdf["cy"]), crs=CRS_UTM,
).to_crs(CRS_GEO)
sectors_gdf["centroid_lon"] = centroid_pts.x
sectors_gdf["centroid_lat"] = centroid_pts.y

for col in ["cx", "cy", "centroid_lon", "centroid_lat", "area_km2"]:
    sectors_geo[col] = sectors_gdf[col].values

print("Centroid offset from tower (m):")
offsets = np.hypot(sectors_gdf["cx"] - sectors_gdf["x"],
                   sectors_gdf["cy"] - sectors_gdf["y"])
print(offsets.describe().to_string())

## Figure: Sector overview map

In [ ]:
fig, ax = plt.subplots(figsize=(10, 12))
comunas.boundary.plot(ax=ax, color="#cccccc", linewidth=0.3)
sectors_geo.plot(
    ax=ax, column="n_sectors", cmap="viridis", alpha=0.4, edgecolor="none",
    legend=True, legend_kwds={"label": "Sectors per BTS", "shrink": 0.5},
    vmin=1, vmax=6,
)
ax.set_title("Sector polygons -- Region Metropolitana")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig(FIGURES / "fig_sectors_r13_overview.pdf", bbox_inches="tight", dpi=150)
plt.show()

## Voronoi tessellation (baseline)

In [ ]:
pts = bts[["x", "y"]].values

bbox = sectors_gdf.total_bounds
margin = 50_000
far_pts = np.array([
    [bbox[0] - margin, bbox[1] - margin],
    [bbox[0] - margin, bbox[3] + margin],
    [bbox[2] + margin, bbox[1] - margin],
    [bbox[2] + margin, bbox[3] + margin],
])
vor = Voronoi(np.vstack([pts, far_pts]))

r13_utm = comunas.to_crs(CRS_UTM).dissolve().geometry.iloc[0]

voronoi_polys = []
for i in range(len(pts)):
    region_idx = vor.point_region[i]
    region = vor.regions[region_idx]
    if -1 in region or len(region) == 0:
        voronoi_polys.append(None)
        continue
    poly = Polygon([vor.vertices[v] for v in region])
    try:
        clipped = poly.intersection(r13_utm)
        voronoi_polys.append(clipped if not clipped.is_empty else None)
    except Exception:
        voronoi_polys.append(None)

bts["voronoi_geom"] = voronoi_polys
bts["voronoi_area_km2"] = [
    g.area / 1e6 if g is not None else np.nan for g in voronoi_polys
]

voronoi_gdf = gpd.GeoDataFrame(
    bts.dropna(subset=["voronoi_geom"]),
    geometry="voronoi_geom", crs=CRS_UTM,
)

## Figure: Resolution gain histogram

In [ ]:
voronoi_res = np.sqrt(bts["voronoi_area_km2"].dropna()) * 1000
sector_res  = np.sqrt(sectors_gdf["area_km2"]) * 1000

fig, ax = plt.subplots(figsize=(6, 4))
bins = np.linspace(0, 10000, 80)
ax.hist(voronoi_res, bins=bins, alpha=0.5, label="Voronoi (per BTS)", color="#e41a1c")
ax.hist(sector_res,  bins=bins, alpha=0.5, label="Sectors",           color="#377eb8")
ax.set_xlabel("Effective resolution $\\sqrt{\\mathrm{area}}$ (m)")
ax.set_ylabel("Count")
ax.set_title("Spatial resolution: sectors vs. Voronoi")
ax.legend()
ax.axvline(voronoi_res.median(), color="#e41a1c", ls=":", lw=1)
ax.axvline(sector_res.median(),  color="#377eb8", ls=":", lw=1)
plt.tight_layout()
plt.savefig(FIGURES / "fig_resolution_gain_histogram.pdf", bbox_inches="tight")
plt.show()

print(f"Median effective resolution:")
print(f"  Voronoi: {voronoi_res.median():,.0f} m")
print(f"  Sectors: {sector_res.median():,.0f} m")
print(f"  Gain:    {voronoi_res.median() / sector_res.median():.1f}x")

## Adaptive H3 cell assignment

In [ ]:
H3_EDGE = {6: 3229, 7: 1221, 8: 461, 9: 174}
H3_RESOLUTIONS = sorted(H3_EDGE.keys())

def pick_h3_resolution(d_nn_m):
    threshold = d_nn_m / 2
    for res in reversed(H3_RESOLUTIONS):
        if H3_EDGE[res] >= threshold:
            return res
    return H3_RESOLUTIONS[0]

bts["h3_res"] = bts["d_nn"].apply(pick_h3_resolution)

def h3_cells_for_sector(geom_utm, h3_res):
    geom_geo = gpd.GeoSeries([geom_utm], crs=CRS_UTM).to_crs(CRS_GEO).iloc[0]
    coords_latlng = [(lat, lng) for lng, lat in geom_geo.exterior.coords]
    try:
        poly = h3.LatLngPoly(coords_latlng)
        cells = h3.polygon_to_cells(poly, h3_res)
    except Exception:
        cells = set()
    if len(cells) == 0:
        c = geom_geo.centroid
        cells = {h3.latlng_to_cell(c.y, c.x, h3_res)}
    return cells

sectors_gdf = sectors_gdf.merge(
    bts[["bts_id", "h3_res"]].drop_duplicates(),
    on="bts_id", how="left", suffixes=("", "_dup"),
)
sectors_gdf = sectors_gdf.drop(
    columns=[c for c in sectors_gdf.columns if c.endswith("_dup")]
)

h3_records = []
for idx, row in sectors_gdf.iterrows():
    h3_cells = h3_cells_for_sector(row.geometry, row["h3_res"])
    for hc in h3_cells:
        h3_records.append({
            "bts_id": row["bts_id"],
            "azimuth": row["azimuth"],
            "h3_index": hc,
            "h3_res": row["h3_res"],
        })

h3_mapping = pd.DataFrame(h3_records)
print(f"H3 mapping: {len(h3_mapping):,} rows, {h3_mapping['h3_index'].nunique():,} unique cells")

## Save outputs

In [ ]:
sectors_save = sectors_geo.drop(columns=["cell_ids"], errors="ignore").copy()
sectors_save.to_parquet(DATA / "sectors.parquet")
print(f"Saved sectors: {len(sectors_save):,} rows")

bts_save = bts.drop(columns=["voronoi_geom"], errors="ignore")
bts_save.to_parquet(DATA / "bts_sites.parquet")
print(f"Saved BTS sites: {len(bts_save):,} rows")

h3_mapping.to_parquet(DATA / "h3_sector_mapping.parquet")
print(f"Saved H3 mapping: {len(h3_mapping):,} rows")

voronoi_save = voronoi_gdf.to_crs(CRS_GEO)
voronoi_save.to_parquet(DATA / "voronoi_bts.parquet")
print(f"Saved Voronoi: {len(voronoi_save):,} rows")

cell_lookup.to_parquet(DATA / "cell_lookup.parquet", index=False)
print(f"Saved cell lookup: {len(cell_lookup):,} rows")